In [1]:
from sedona.spark import SedonaContext
from pyspark.sql import SparkSession

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1794925904908_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
## LOAD FUNCTIONS
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import ArrayType, DoubleType

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
## LOAD CSV FILE
TRIPS_CSV   = "s3://workshop-sedona-tmp/data/porto/porto_taxi.csv"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
trips = (spark.read
         .option("header", "true")
         .option("quote", '"')
         .option("escape", '"')          # POLYLINE contains commas inside quotes
         .option("multiLine", "false")
         .csv(TRIPS_CSV))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
## VIZ THE PORTO TAXI DATASET
print(f"Total raw trips: {trips.count():,}")
trips.show(5, truncate=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total raw trips: 1,710,670
+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|            TRIP_ID|CALL_TYPE|ORIGIN_CALL|ORIGIN_STAND| TAXI_ID| TIMESTAMP|DAY_TYPE|MISSING_DATA|            POLYLINE|
+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|1372636858620000589|        C|       NULL|        NULL|20000589|1372636858|       A|       False|[[-8.618643,41.14...|
|1372637303620000596|        B|       NULL|           7|20000596|1372637303|       A|       False|[[-8.639847,41.15...|
|1372636951620000320|        C|       NULL|        NULL|20000320|1372636951|       A|       False|[[-8.612964,41.14...|
|1372636854620000520|        C|       NULL|        NULL|20000520|1372636854|       A|       False|[[-8.574678,41.15...|
|1372637091620000337|        C|       NULL|        NULL|20000337|1372637091|       A|       False|[[-8.645994,41.18...|
+------------

In [8]:
## Convert the array into geometry objects 
df_parsed = trips.withColumn(
    "points",from_json(col("POLYLINE"), ArrayType(ArrayType(DoubleType())))
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
df_parsed.createOrReplaceTempView("trips_parsed")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
## Filter noise: empty polylines, too-short, missing-data flag ----
## NOTE: in the Kaggle CSV, MISSING_DATA is the string 'False' / 'True'
df_porto_bronze = spark.sql("""
SELECT
    TRIP_ID,
    TAXI_ID,
    CAST(TIMESTAMP AS LONG) AS trip_ts,
    CALL_TYPE,
    points
FROM trips_parsed
WHERE points IS NOT NULL
  AND size(points) >= 2              -- need ≥2 points to form a route
  AND MISSING_DATA = 'False'         -- drop flagged gap trips
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
## Verify new totals
total = trips.count()
total_bronze  = df_porto_bronze.count()
print("Total CSV: " + str(total))
print("Total RAW: " + str(total_bronze))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total CSV: 1710670
Total RAW: 1674152

In [12]:
## WRITE INTO 1-LAYER LAKE
df_porto_bronze.write.mode("overwrite").parquet("s3://workshop-sedona-datalake/bronze/porto_trips_parsed.parquet")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…